In [1]:
import sys; print(sys.executable)

/home/hasnain/projects/velo-disponibility/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [4]:
df = pd.read_csv('../data/raw/readings.csv')
df.head()

,fetched_at,lastupdate,idstation,nom,lon,lat,etat,slots_total,slots_free,bikes,consistent
0,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5505,Sainte-Anne,-1.680534,48.114219,En fonctionnement,24,13,11,1
1,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5509,Saint-Georges Piscine,-1.674417,48.112385,En fonctionnement,18,16,2,1
2,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5510,Musée Beaux-Arts,-1.674080,48.109601,En fonctionnement,15,1,14,1
3,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5514,Bonnets Rouges,-1.665814,48.106846,En fonctionnement,23,18,5,1
4,2026-10-09T17:24:25+00:00,2026-10-09T17:23:08+00:00,5517,Charles de Gaulle,-1.677119,48.105111,En fonctionnement,24,8,16,1


# 1. Filter active stations and valid records

In [5]:
df = df[df['etat'] == 'En fonctionnement'].copy()
df = df[df['consistent'] == 1]

# 2. Check Duplicates

In [6]:
print(df.duplicated(["idstation", "lastupdate"]).sum())

0


# 3. Parse timestamps

In [7]:
df['lastupdate'] = pd.to_datetime(df['lastupdate'])

# 4. Create target classes

In [8]:
# Define bin edges: (-inf to 0], (0 to 2], (2 to 5], (5 to inf)
bins = [-1, 0, 2, 5, float('inf')]
labels = [0, 1, 2, 3]  # Class 0: Empty/Full, 1: Critical, 2: Moderate, 3: High

# Apply directly to columns
df['bike_bin'] = pd.cut(df['bikes'], bins=bins, labels=labels).astype(int)

# 5. Feature engineering: Extract temporal features

In [9]:
df['ts'] = pd.to_datetime(df['lastupdate'], utc=True).dt.tz_convert('Europe/Paris')
df['hour'] = df['ts'].dt.hour
df['day_of_week'] = df['ts'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

/tmp/ipykernel_79259/3469264913.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['ts'].dt.dayofweek


# 6. Select features for ML

In [10]:
feature_cols = ['idstation', 'lat', 'lon', 'slots_total', 'hour', 'day_of_week', 'is_weekend']
target_col = ['bike_bin']

In [11]:
print(df['bike_bin'].value_counts())
print(df['bike_bin'].value_counts(normalize=True).round(2))

bike_bin
3    8108
2    2805
1    1484
0     535
Name: count, dtype: int64
bike_bin
3    0.63
2    0.22
1    0.11
0    0.04
Name: proportion, dtype: float64


In [12]:
train, test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['bike_bin']
)

In [13]:
def mode(s): return s.mode().iloc[0]

# A: always the most common class
major = train['bike_bin'].mode()[0]
pred_a = pd.Series(major, index=test.index)

# B: most common class per station + hour + weekday, with fallbacks
keys = [['idstation','hour','day_of_week'], ['idstation','hour'], ['idstation']]
pred_b = pd.Series(np.nan, index=test.index)
for k in keys:
    table = train.groupby(k)['bike_bin'].agg(mode).rename('p').reset_index()
    m = test[k].merge(table, on=k, how='left')
    m.index = test.index
    pred_b = pred_b.fillna(m['p'])
pred_b = pred_b.fillna(major).astype(int)

for name, p in [('majority', pred_a), ('station-hour-weekday', pred_b)]:
    print(name, accuracy_score(test['bike_bin'], p),
          f1_score(test['bike_bin'], p, average='macro'))

majority 0.6269810591418632 0.19268234735091472
station-hour-weekday 0.6903749516814843 0.5518820637747148


In [16]:
df
train

,fetched_at,lastupdate,idstation,nom,lon,lat,etat,slots_total,slots_free,bikes,consistent,bike_bin,ts,hour,day_of_week,is_weekend
11867,2026-10-21T10:40:00+00:00,2026-10-21 10:38:09+00:00,5548,Beaulieu - Université,-1.639922,48.121926,En fonctionnement,29,22,7,1,3,2026-10-21 12:38:09+02:00,12,2,0
3066,2026-10-12T14:00:00+00:00,2026-10-12 13:58:07+00:00,5539,Pontchaillou,-1.692737,48.121481,En fonctionnement,27,19,8,1,3,2026-10-12 15:58:07+02:00,15,0,0
6885,2026-10-15T10:20:00+00:00,2026-10-15 10:18:08+00:00,5539,Pontchaillou,-1.692737,48.121481,En fonctionnement,28,14,14,1,3,2026-10-15 12:18:08+02:00,12,3,0
13008,2026-10-22T15:40:00+00:00,2026-10-22 15:37:08+00:00,5566,Bréquigny Piscine,-1.690340,48.089986,En fonctionnement,24,18,6,1,3,2026-10-22 17:37:08+02:00,17,3,0
3530,2026-10-13T10:40:00+00:00,2026-10-13 10:38:09+00:00,5578,Gros-Chêne,-1.664806,48.125408,En fonctionnement,18,13,5,1,2,2026-10-13 12:38:09+02:00,12,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5657,2026-10-14T14:40:00+00:00,2026-10-14 14:38:08+00:00,5580,Painlevé,-1.659762,48.123463,En fonctionnement,29,27,2,1,1,2026-10-14 16:38:08+02:00,16,2,0
8103,2026-10-16T15:20:00+00:00,2026-10-16 15:18:08+00:00,5540,Croix Saint-Hélier,-1.662078,48.103044,En fonctionnement,19,13,6,1,3,2026-10-16 17:18:08+02:00,17,4,0
535,2026-10-10T11:00:00+00:00,2026-10-10 10:58:08+00:00,5523,La Rotonde,-1.686770,48.106653,En fonctionnement,19,10,9,1,3,2026-10-10 12:58:08+02:00,12,5,1
101,2026-10-09T17:44:25+00:00,2026-10-09 17:43:08+00:00,5538,Marbeuf,-1.702375,48.111825,En fonctionnement,20,17,3,1,2,2026-10-09 19:43:08+02:00,19,4,0


In [ ]:
from pathlib import Path

output_dir = Path('../data')
output_dir.mkdir(parents=True, exist_ok=True)
df.to_parquet(output_dir / 'clean.parquet', index=False)
train.to_parquet(output_dir / 'train.parquet', index=False)
test.to_parquet(output_dir / 'test.parquet', index=False)
print(f'Saved {len(df):,} cleaned rows, {len(train):,} train rows, and {len(test):,} test rows.')